# 기본 베이스라인

In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# 파일 경로 설정부
file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_view_delete\Membership_v2.csv"

# 사용 컬럼 설정부
use_cols = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
    "is_repurchase",
]

# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)

# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )

# 데이터 로드부
df = pd.read_csv(file_path, usecols=use_cols).copy()

# 숫자형 변환부
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["max_screen"] = pd.to_numeric(df["max_screen"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수, 타깃 변수 생성부
X = df[
    [
        "price",
        "max_screen",
        "is_promotion",
        "is_churn_prevented",
        "payment_device",
        "is_user_verified",
        "gender",
        "age",
    ]
].copy()

# 양성 클래스 정의부
# is_repurchase == 0 을 예측 목표로 두기 때문에 0이면 1, 1이면 0으로 변환
y = (df["is_repurchase_num"] == 0).astype(int)

# 숫자형, 범주형 컬럼 구분부
numeric_features = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
    "age",
]

categorical_features = [
    "payment_device",
    "gender",
]

# 전처리 파이프라인 구성부
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 모델 정의부
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42,
    ),
}

# 평가 수행부
results = []

for model_name, model in models.items():
    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    result = {
        "model": model_name,
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    }

    results.append(result)

# 결과 출력부
results_df = (
    pd.DataFrame(results)
    .set_index("model")
    [["precision", "recall", "f1_score", "roc_auc", "pr_auc"]]
    .round(4)
    .sort_values("f1_score", ascending=False)
)

print("양성 클래스 기준: is_repurchase == 0")
print(results_df)

양성 클래스 기준: is_repurchase == 0
                    precision  recall  f1_score  roc_auc  pr_auc
model                                                           
LogisticRegression     0.3379  0.5768    0.4261   0.5800  0.3393
RandomForest           0.3337  0.5196    0.4064   0.5615  0.3290
GradientBoosting       0.3333  0.0008    0.0015   0.5873  0.3460


# 파생 추가

In [4]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# 파일 경로 설정부
file_paths = [
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_0.csv",
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_1.csv",
]

# 기존 사용 컬럼 우선순위 설정부
base_feature_candidates = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
]

# 제외 컬럼 설정부
exclude_cols = {
    "USER_NUM",
    "USER_KEY",
    "reg_date",
    "end_date",
    "is_repurchase",
    "is_repurchase_num",
}

# 범주형 컬럼 후보 설정부
categorical_candidates = {
    "payment_device",
    "gender",
}


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# 데이터 로드 및 병합부
df_list = [pd.read_csv(path) for path in file_paths]
df = pd.concat(df_list, ignore_index=True).copy()

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    if col in df.columns:
        df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 숫자형 변환부
for col in df.columns:
    if col in categorical_candidates or col in {"USER_KEY", "reg_date", "end_date"}:
        continue

    df[col] = pd.to_numeric(df[col], errors="coerce")

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수 컬럼 구성부
base_features = [
    col for col in base_feature_candidates
    if col in df.columns and col not in exclude_cols
]

extra_features = [
    col for col in df.columns
    if col not in exclude_cols and col not in base_features
]

feature_cols = base_features + extra_features

if not feature_cols:
    raise ValueError("사용 가능한 입력 변수 컬럼이 없습니다.")

# 입력 변수, 타깃 변수 생성부
X = df[feature_cols].copy()

# 양성 클래스 정의부
# is_repurchase == 0 예측 목표 설정부
y = (df["is_repurchase_num"] == 0).astype(int)

# 숫자형, 범주형 컬럼 구분부
categorical_features = [
    col for col in feature_cols
    if col in categorical_candidates and col in X.columns
]

numeric_features = [
    col for col in feature_cols
    if col not in categorical_features
]

# 전처리 파이프라인 구성부
transformers = []

if numeric_features:
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    transformers.append(("num", numeric_transformer, numeric_features))

if categorical_features:
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )
    transformers.append(("cat", categorical_transformer, categorical_features))

preprocessor = ColumnTransformer(transformers=transformers)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 모델 정의부
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42,
    ),
}

# 평가 수행부
results = []

for model_name, model in models.items():
    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    result = {
        "model": model_name,
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    }

    results.append(result)

# 결과 출력부
results_df = (
    pd.DataFrame(results)
    .set_index("model")
    [["precision", "recall", "f1_score", "roc_auc", "pr_auc"]]
    .round(4)
    .sort_values("f1_score", ascending=False)
)

print("양성 클래스 기준: is_repurchase == 0")
print(results_df)


양성 클래스 기준: is_repurchase == 0
                    precision  recall  f1_score  roc_auc  pr_auc
model                                                           
GradientBoosting       0.7587  0.6694    0.7112   0.9065  0.7720
RandomForest           0.7518  0.6669    0.7068   0.8986  0.7594
LogisticRegression     0.6242  0.7984    0.7006   0.8836  0.7387


In [5]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# 파일 경로 설정부
file_paths = [
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_0.csv",
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_1.csv",
]

# 기존 사용 컬럼 우선순위 설정부
base_feature_candidates = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
]

# 제외 컬럼 설정부
exclude_cols = {
    "USER_NUM",
    "USER_KEY",
    "reg_date",
    "end_date",
    "is_repurchase",
    "is_repurchase_num",
}

# 범주형 컬럼 후보 설정부
categorical_candidates = {
    "payment_device",
    "gender",
}

# VIF 기준 설정부
vif_threshold = 10.0


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# VIF 계산 함수부
def calculate_vif_table(df):
    rows = []

    for target_col in df.columns:
        x_cols = [col for col in df.columns if col != target_col]

        if not x_cols:
            vif_value = 1.0
        else:
            x = df[x_cols]
            y = df[target_col]

            model = LinearRegression()
            model.fit(x, y)
            r2 = model.score(x, y)

            if r2 >= 0.999999:
                vif_value = np.inf
            else:
                vif_value = 1.0 / (1.0 - r2)

        rows.append(
            {
                "feature": target_col,
                "vif": vif_value,
            }
        )

    return pd.DataFrame(rows).sort_values("vif", ascending=False).reset_index(drop=True)


# VIF 기반 제거 함수부
def remove_high_vif_features(df, threshold):
    working = df.copy()
    removed_rows = []

    while working.shape[1] > 1:
        vif_df = calculate_vif_table(working)
        max_vif = vif_df["vif"].iloc[0]

        if pd.isna(max_vif) or max_vif < threshold:
            break

        drop_feature = vif_df.iloc[0]["feature"]
        drop_vif = vif_df.iloc[0]["vif"]

        removed_rows.append(
            {
                "drop_feature": drop_feature,
                "vif": drop_vif,
            }
        )

        working = working.drop(columns=[drop_feature])

    removed_df = pd.DataFrame(removed_rows)
    final_vif_df = calculate_vif_table(working)

    return working, removed_df, final_vif_df


# 전처리기 생성 함수부
def build_preprocessor(feature_cols, categorical_candidates):
    categorical_features = [
        col for col in feature_cols
        if col in categorical_candidates
    ]

    numeric_features = [
        col for col in feature_cols
        if col not in categorical_features
    ]

    transformers = []

    if numeric_features:
        numeric_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]
        )
        transformers.append(("num", numeric_transformer, numeric_features))

    if categorical_features:
        categorical_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_onehot_encoder()),
            ]
        )
        transformers.append(("cat", categorical_transformer, categorical_features))

    return ColumnTransformer(transformers=transformers)


# 모델 평가 함수부
def evaluate_models(X_train, X_test, y_train, y_test, feature_cols, categorical_candidates):
    preprocessor = build_preprocessor(
        feature_cols=feature_cols,
        categorical_candidates=categorical_candidates,
    )

    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(
            random_state=42,
        ),
    }

    results = []

    for model_name, model in models.items():
        clf = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", model),
            ]
        )

        clf.fit(X_train[feature_cols], y_train)

        y_pred = clf.predict(X_test[feature_cols])
        y_proba = clf.predict_proba(X_test[feature_cols])[:, 1]

        results.append(
            {
                "model": model_name,
                "precision": precision_score(y_test, y_pred, zero_division=0),
                "recall": recall_score(y_test, y_pred, zero_division=0),
                "f1_score": f1_score(y_test, y_pred, zero_division=0),
                "roc_auc": roc_auc_score(y_test, y_proba),
                "pr_auc": average_precision_score(y_test, y_proba),
            }
        )

    return (
        pd.DataFrame(results)
        .set_index("model")
        [["precision", "recall", "f1_score", "roc_auc", "pr_auc"]]
        .round(4)
        .sort_values("f1_score", ascending=False)
    )


# 데이터 로드 및 병합부
df_list = [pd.read_csv(path) for path in file_paths]
df = pd.concat(df_list, ignore_index=True).copy()

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    if col in df.columns:
        df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 숫자형 변환부
for col in df.columns:
    if col in categorical_candidates or col in {"USER_KEY", "reg_date", "end_date"}:
        continue

    df[col] = pd.to_numeric(df[col], errors="coerce")

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수 컬럼 구성부
base_features = [
    col for col in base_feature_candidates
    if col in df.columns and col not in exclude_cols
]

extra_features = [
    col for col in df.columns
    if col not in exclude_cols and col not in base_features
]

feature_cols = base_features + extra_features

if not feature_cols:
    raise ValueError("사용 가능한 입력 변수 컬럼이 없습니다.")

# 입력 변수, 타깃 변수 생성부
X = df[feature_cols].copy()

# 양성 클래스 정의부
# is_repurchase == 0 예측 목표 설정부
y = (df["is_repurchase_num"] == 0).astype(int)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 숫자형, 범주형 컬럼 구분부
categorical_features = [
    col for col in feature_cols
    if col in categorical_candidates
]

numeric_features = [
    col for col in feature_cols
    if col not in categorical_features
]

# VIF 계산용 숫자형 데이터 준비부
numeric_imputer = SimpleImputer(strategy="median")
X_train_numeric = pd.DataFrame(
    numeric_imputer.fit_transform(X_train[numeric_features]),
    columns=numeric_features,
    index=X_train.index,
)

# VIF 기반 다중공선성 제거부
reduced_numeric_df, removed_vif_df, final_vif_df = remove_high_vif_features(
    X_train_numeric,
    threshold=vif_threshold,
)

final_numeric_features = reduced_numeric_df.columns.tolist()
final_feature_cols = final_numeric_features + categorical_features

# 제거 전 성능 평가부
baseline_results_df = evaluate_models(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=feature_cols,
    categorical_candidates=categorical_candidates,
)

# 제거 후 성능 평가부
reduced_results_df = evaluate_models(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=final_feature_cols,
    categorical_candidates=categorical_candidates,
)

# 성능 비교표 생성부
comparison_df = pd.concat(
    [
        baseline_results_df.assign(scenario="before_vif"),
        reduced_results_df.assign(scenario="after_vif"),
    ]
).reset_index()

comparison_df = comparison_df[
    ["scenario", "model", "precision", "recall", "f1_score", "roc_auc", "pr_auc"]
]

# 출력부
print("양성 클래스 기준: is_repurchase == 0")
print(f"원래 사용 컬럼 수: {len(feature_cols)}")
print(f"VIF로 제거된 숫자형 컬럼 수: {len(removed_vif_df)}")
print(f"최종 사용 컬럼 수: {len(final_feature_cols)}")
print()

print("제거 전 모델 성능")
print(baseline_results_df)
print()

print("제거 후 모델 성능")
print(reduced_results_df)
print()

print("제거 전후 성능 비교")
print(comparison_df.to_string(index=False))


양성 클래스 기준: is_repurchase == 0
원래 사용 컬럼 수: 134
VIF로 제거된 숫자형 컬럼 수: 61
최종 사용 컬럼 수: 73

제거 전 모델 성능
                    precision  recall  f1_score  roc_auc  pr_auc
model                                                           
GradientBoosting       0.7587  0.6694    0.7112   0.9065  0.7720
RandomForest           0.7518  0.6669    0.7068   0.8986  0.7594
LogisticRegression     0.6242  0.7984    0.7006   0.8836  0.7387

제거 후 모델 성능
                    precision  recall  f1_score  roc_auc  pr_auc
model                                                           
GradientBoosting       0.7671  0.6403    0.6980   0.8973  0.7576
RandomForest           0.7470  0.6500    0.6951   0.8909  0.7538
LogisticRegression     0.6130  0.7855    0.6886   0.8790  0.7272

제거 전후 성능 비교
  scenario              model  precision  recall  f1_score  roc_auc  pr_auc
before_vif   GradientBoosting     0.7587  0.6694    0.7112   0.9065  0.7720
before_vif       RandomForest     0.7518  0.6669    0.7068   0.8986  0.7594
be

## 데이터 누수 확인

In [10]:
import re
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# 파일 경로 설정부
file_paths = [
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_0.csv",
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_1.csv",
]

# 기존 사용 컬럼 우선순위 설정부
base_feature_candidates = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
]

# 제외 컬럼 설정부
exclude_cols = {
    "USER_NUM",
    "USER_KEY",
    "reg_date",
    "end_date",
    "is_repurchase",
    "is_repurchase_num",
}

# 범주형 컬럼 후보 설정부
categorical_candidates = {
    "payment_device",
    "gender",
}

# VIF 기준 설정부
vif_threshold = 10.0


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# VIF 계산 함수부
def calculate_vif_table(df):
    rows = []

    for target_col in df.columns:
        x_cols = [col for col in df.columns if col != target_col]

        if not x_cols:
            vif_value = 1.0
        else:
            x = df[x_cols]
            y = df[target_col]

            model = LinearRegression()
            model.fit(x, y)
            r2 = model.score(x, y)

            if r2 >= 0.999999:
                vif_value = np.inf
            else:
                vif_value = 1.0 / (1.0 - r2)

        rows.append(
            {
                "feature": target_col,
                "vif": vif_value,
            }
        )

    return pd.DataFrame(rows).sort_values("vif", ascending=False).reset_index(drop=True)


# VIF 기반 제거 함수부
def remove_high_vif_features(df, threshold):
    working = df.copy()

    while working.shape[1] > 1:
        vif_df = calculate_vif_table(working)
        max_vif = vif_df["vif"].iloc[0]

        if pd.isna(max_vif) or max_vif < threshold:
            break

        drop_feature = vif_df.iloc[0]["feature"]
        working = working.drop(columns=[drop_feature])

    final_vif_df = calculate_vif_table(working)

    return working, final_vif_df


# 전처리기 생성 함수부
def build_preprocessor(feature_cols, categorical_candidates):
    categorical_features = [
        col for col in feature_cols
        if col in categorical_candidates
    ]

    numeric_features = [
        col for col in feature_cols
        if col not in categorical_features
    ]

    transformers = []

    if numeric_features:
        numeric_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]
        )
        transformers.append(("num", numeric_transformer, numeric_features))

    if categorical_features:
        categorical_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_onehot_encoder()),
            ]
        )
        transformers.append(("cat", categorical_transformer, categorical_features))

    return ColumnTransformer(transformers=transformers)


# 공통 지표 계산 함수부
def calculate_metrics(y_true, y_pred, y_proba):
    return {
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }


# 과적합 판정 함수부
def judge_overfitting(train_f1, test_f1, train_roc_auc, test_roc_auc):
    f1_gap = train_f1 - test_f1
    roc_auc_gap = train_roc_auc - test_roc_auc

    if f1_gap >= 0.05 or roc_auc_gap >= 0.05:
        return "의심"

    return "낮음"


# 모델 평가 함수부
def evaluate_models(X_train, X_test, y_train, y_test, feature_cols, categorical_candidates):
    preprocessor = build_preprocessor(
        feature_cols=feature_cols,
        categorical_candidates=categorical_candidates,
    )

    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(
            random_state=42,
        ),
    }

    results = []
    fitted_models = {}

    for model_name, model in models.items():
        clf = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", model),
            ]
        )

        clf.fit(X_train[feature_cols], y_train)

        y_train_pred = clf.predict(X_train[feature_cols])
        y_train_proba = clf.predict_proba(X_train[feature_cols])[:, 1]

        y_test_pred = clf.predict(X_test[feature_cols])
        y_test_proba = clf.predict_proba(X_test[feature_cols])[:, 1]

        train_metrics = calculate_metrics(y_train, y_train_pred, y_train_proba)
        test_metrics = calculate_metrics(y_test, y_test_pred, y_test_proba)

        results.append(
            {
                "model": model_name,
                "train_f1": train_metrics["f1_score"],
                "test_f1": test_metrics["f1_score"],
                "train_roc_auc": train_metrics["roc_auc"],
                "test_roc_auc": test_metrics["roc_auc"],
                "overfitting": judge_overfitting(
                    train_f1=train_metrics["f1_score"],
                    test_f1=test_metrics["f1_score"],
                    train_roc_auc=train_metrics["roc_auc"],
                    test_roc_auc=test_metrics["roc_auc"],
                ),
            }
        )

        fitted_models[model_name] = clf

    results_df = (
        pd.DataFrame(results)
        .set_index("model")
        .round(4)
        .sort_values("test_f1", ascending=False)
    )

    return results_df, fitted_models


# 누수 의심 컬럼 개수 계산 함수부
def count_suspicious_features(feature_cols):
    suspicious_patterns = {
        "repurchase": r"(^|_)repurchase($|_)",
        "churn": r"(^|_)churn($|_)",
        "last": r"(^|_)last($|_)",
        "end": r"(^|_)end($|_)",
        "gap": r"(^|_)gap($|_)",
        "retention": r"(^|_)retention($|_)",
        "recency": r"(^|_)recency($|_)",
        "late": r"(^|_)late($|_)",
        "week4": r"(^|_)week4($|_)",
        "week5": r"(^|_)week5($|_)",
    }

    suspicious_count = 0

    for feature in feature_cols:
        lowered = feature.lower()

        matched = any(
            re.search(pattern, lowered)
            for pattern in suspicious_patterns.values()
        )

        if matched:
            suspicious_count += 1

    return suspicious_count


# 셔플 타깃 점검 함수부
def check_shuffled_target(X_train, X_test, y_train, y_test, feature_cols, categorical_candidates, model):
    shuffled_y_train = pd.Series(
        np.random.RandomState(42).permutation(y_train.to_numpy()),
        index=y_train.index,
    )

    preprocessor = build_preprocessor(
        feature_cols=feature_cols,
        categorical_candidates=categorical_candidates,
    )

    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", clone(model)),
        ]
    )

    clf.fit(X_train[feature_cols], shuffled_y_train)
    y_test_proba = clf.predict_proba(X_test[feature_cols])[:, 1]

    return roc_auc_score(y_test, y_test_proba)


# 데이터 누수 판정 함수부
def judge_leakage(shuffled_roc_auc, suspicious_feature_count):
    if shuffled_roc_auc >= 0.60:
        return "의심"

    if suspicious_feature_count > 0:
        return "컬럼 점검 필요"

    return "낮음"


# 데이터 로드 및 병합부
df_list = [pd.read_csv(path) for path in file_paths]
df = pd.concat(df_list, ignore_index=True).copy()

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    if col in df.columns:
        df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 숫자형 변환부
for col in df.columns:
    if col in categorical_candidates or col in {"USER_KEY", "reg_date", "end_date"}:
        continue

    df[col] = pd.to_numeric(df[col], errors="coerce")

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수 컬럼 구성부
base_features = [
    col for col in base_feature_candidates
    if col in df.columns and col not in exclude_cols
]

extra_features = [
    col for col in df.columns
    if col not in exclude_cols and col not in base_features
]

feature_cols = base_features + extra_features

if not feature_cols:
    raise ValueError("사용 가능한 입력 변수 컬럼이 없습니다.")

# 입력 변수, 타깃 변수 생성부
X = df[feature_cols].copy()

# 양성 클래스 정의부
# is_repurchase == 0 예측 목표 설정부
y = (df["is_repurchase_num"] == 0).astype(int)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 숫자형, 범주형 컬럼 구분부
categorical_features = [
    col for col in feature_cols
    if col in categorical_candidates
]

numeric_features = [
    col for col in feature_cols
    if col not in categorical_features
]

# VIF 계산용 숫자형 데이터 준비부
if numeric_features:
    numeric_imputer = SimpleImputer(strategy="median")
    X_train_numeric = pd.DataFrame(
        numeric_imputer.fit_transform(X_train[numeric_features]),
        columns=numeric_features,
        index=X_train.index,
    )

    reduced_numeric_df, final_vif_df = remove_high_vif_features(
        X_train_numeric,
        threshold=vif_threshold,
    )

    final_numeric_features = reduced_numeric_df.columns.tolist()
else:
    final_vif_df = pd.DataFrame(columns=["feature", "vif"])
    final_numeric_features = []

# 최종 입력 변수 구성부
final_feature_cols = final_numeric_features + categorical_features

if not final_feature_cols:
    raise ValueError("최종 사용 가능한 입력 변수 컬럼이 없습니다.")

# 최종 변수 기준 모델 평가부
results_df, fitted_models = evaluate_models(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=final_feature_cols,
    categorical_candidates=categorical_candidates,
)

# 최고 성능 모델 선택부
best_model_name = results_df.index[0]
best_pipeline = fitted_models[best_model_name]
best_model = best_pipeline.named_steps["model"]

# 누수 의심 컬럼 개수 계산부
suspicious_feature_count = count_suspicious_features(final_feature_cols)

# 셔플 타깃 점검부
shuffled_roc_auc = check_shuffled_target(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=final_feature_cols,
    categorical_candidates=categorical_candidates,
    model=best_model,
)

# 데이터 누수 판정부
leakage_judgment = judge_leakage(
    shuffled_roc_auc=shuffled_roc_auc,
    suspicious_feature_count=suspicious_feature_count,
)

# 최고 성능 모델 과적합 판정값 추출부
best_model_overfitting = results_df.loc[best_model_name, "overfitting"]

# 출력부
print("양성 클래스 기준: is_repurchase == 0")
print(f"최종 사용 컬럼 수: {len(final_feature_cols)}")
print()
print("모델별 Train / Test 점수")
print(results_df.to_string())
print()
print(f"최고 성능 모델: {best_model_name}")
print(f"최고 성능 모델 과적합 여부: {best_model_overfitting}")
print(f"데이터 누수 여부: {leakage_judgment}")
print(f"셔플 타깃 ROC AUC: {shuffled_roc_auc:.4f}")
print(f"누수 의심 컬럼 수: {suspicious_feature_count}")


양성 클래스 기준: is_repurchase == 0
최종 사용 컬럼 수: 73

모델별 Train / Test 점수
                    train_f1  test_f1  train_roc_auc  test_roc_auc overfitting
model                                                                         
GradientBoosting      0.7098   0.6980         0.9105        0.8973          낮음
RandomForest          0.9740   0.6951         0.9984        0.8909          의심
LogisticRegression    0.6849   0.6886         0.8712        0.8790          낮음

최고 성능 모델: GradientBoosting
최고 성능 모델 과적합 여부: 낮음
데이터 누수 여부: 컬럼 점검 필요
셔플 타깃 ROC AUC: 0.4957
누수 의심 컬럼 수: 11


## 데이터 누수 의심 컬럼 제거

In [13]:
import re
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# 파일 경로 설정부
file_paths = [
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_0.csv",
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_1.csv",
]

# 기존 사용 컬럼 우선순위 설정부
base_feature_candidates = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
]

# 제외 컬럼 설정부
exclude_cols = {
    "USER_NUM",
    "USER_KEY",
    "reg_date",
    "end_date",
    "is_repurchase",
    "is_repurchase_num",
}

# 범주형 컬럼 후보 설정부
categorical_candidates = {
    "payment_device",
    "gender",
}

# VIF 기준 설정부
vif_threshold = 10.0

# 최적 모델 선택 기준 설정부
acceptable_roc_gap = 0.03


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# VIF 계산 함수부
def calculate_vif_table(df):
    rows = []

    for target_col in df.columns:
        x_cols = [col for col in df.columns if col != target_col]

        if not x_cols:
            vif_value = 1.0
        else:
            x = df[x_cols]
            y = df[target_col]

            model = LinearRegression()
            model.fit(x, y)
            r2 = model.score(x, y)

            if r2 >= 0.999999:
                vif_value = np.inf
            else:
                vif_value = 1.0 / (1.0 - r2)

        rows.append(
            {
                "feature": target_col,
                "vif": vif_value,
            }
        )

    return pd.DataFrame(rows).sort_values("vif", ascending=False).reset_index(drop=True)


# VIF 기반 제거 함수부
def remove_high_vif_features(df, threshold):
    working = df.copy()

    while working.shape[1] > 1:
        vif_df = calculate_vif_table(working)
        max_vif = vif_df["vif"].iloc[0]

        if pd.isna(max_vif) or max_vif < threshold:
            break

        drop_feature = vif_df.iloc[0]["feature"]
        working = working.drop(columns=[drop_feature])

    final_vif_df = calculate_vif_table(working)

    return working, final_vif_df


# 전처리기 생성 함수부
def build_preprocessor(feature_cols, categorical_candidates):
    categorical_features = [
        col for col in feature_cols
        if col in categorical_candidates
    ]

    numeric_features = [
        col for col in feature_cols
        if col not in categorical_features
    ]

    transformers = []

    if numeric_features:
        numeric_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]
        )
        transformers.append(("num", numeric_transformer, numeric_features))

    if categorical_features:
        categorical_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_onehot_encoder()),
            ]
        )
        transformers.append(("cat", categorical_transformer, categorical_features))

    return ColumnTransformer(transformers=transformers)


# 공통 지표 계산 함수부
def calculate_metrics(y_true, y_pred, y_proba):
    return {
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }


# 과적합 판정 함수부
def judge_overfitting(train_f1, test_f1, train_roc_auc, test_roc_auc):
    f1_gap = train_f1 - test_f1
    roc_auc_gap = train_roc_auc - test_roc_auc

    if f1_gap >= 0.05 or roc_auc_gap >= 0.05:
        return "의심"

    return "낮음"


# 모델 평가 함수부
def evaluate_models(X_train, X_test, y_train, y_test, feature_cols, categorical_candidates):
    preprocessor = build_preprocessor(
        feature_cols=feature_cols,
        categorical_candidates=categorical_candidates,
    )

    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(
            random_state=42,
        ),
    }

    results = []
    fitted_models = {}

    for model_name, model in models.items():
        clf = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", model),
            ]
        )

        clf.fit(X_train[feature_cols], y_train)

        y_train_pred = clf.predict(X_train[feature_cols])
        y_train_proba = clf.predict_proba(X_train[feature_cols])[:, 1]

        y_test_pred = clf.predict(X_test[feature_cols])
        y_test_proba = clf.predict_proba(X_test[feature_cols])[:, 1]

        train_metrics = calculate_metrics(y_train, y_train_pred, y_train_proba)
        test_metrics = calculate_metrics(y_test, y_test_pred, y_test_proba)

        results.append(
            {
                "model": model_name,
                "train_f1": train_metrics["f1_score"],
                "test_f1": test_metrics["f1_score"],
                "gap_f1": train_metrics["f1_score"] - test_metrics["f1_score"],
                "train_roc_auc": train_metrics["roc_auc"],
                "test_roc_auc": test_metrics["roc_auc"],
                "gap_roc_auc": train_metrics["roc_auc"] - test_metrics["roc_auc"],
                "overfitting": judge_overfitting(
                    train_f1=train_metrics["f1_score"],
                    test_f1=test_metrics["f1_score"],
                    train_roc_auc=train_metrics["roc_auc"],
                    test_roc_auc=test_metrics["roc_auc"],
                ),
            }
        )

        fitted_models[model_name] = clf

    results_df = pd.DataFrame(results).set_index("model").round(4)

    return results_df, fitted_models


# 누수 의심 컬럼 추출 함수부
def get_suspicious_features(feature_cols):
    suspicious_patterns = {
        "repurchase": r"(^|_)repurchase($|_)",
        "churn": r"(^|_)churn($|_)",
        "last": r"(^|_)last($|_)",
        "end": r"(^|_)end($|_)",
        "gap": r"(^|_)gap($|_)",
        "retention": r"(^|_)retention($|_)",
        "recency": r"(^|_)recency($|_)",
        "late": r"(^|_)late($|_)",
        "week4": r"(^|_)week4($|_)",
        "week5": r"(^|_)week5($|_)",
    }

    suspicious_features = []

    for feature in feature_cols:
        lowered = feature.lower()

        matched = any(
            re.search(pattern, lowered)
            for pattern in suspicious_patterns.values()
        )

        if matched:
            suspicious_features.append(feature)

    return suspicious_features


# 셔플 타깃 점검 함수부
def check_shuffled_target(X_train, X_test, y_train, y_test, feature_cols, categorical_candidates, model):
    shuffled_y_train = pd.Series(
        np.random.RandomState(42).permutation(y_train.to_numpy()),
        index=y_train.index,
    )

    preprocessor = build_preprocessor(
        feature_cols=feature_cols,
        categorical_candidates=categorical_candidates,
    )

    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", clone(model)),
        ]
    )

    clf.fit(X_train[feature_cols], shuffled_y_train)
    y_test_proba = clf.predict_proba(X_test[feature_cols])[:, 1]

    return roc_auc_score(y_test, y_test_proba)


# 데이터 누수 판정 함수부
def judge_leakage(shuffled_roc_auc, suspicious_feature_count):
    if shuffled_roc_auc >= 0.60:
        return "의심"

    if shuffled_roc_auc >= 0.55:
        return "약간 의심"

    if suspicious_feature_count > 0:
        return "컬럼 점검 필요"

    return "낮음"


# 최적 모델 선택 함수부
def select_best_model(results_df):
    non_overfit_df = results_df[results_df["overfitting"] == "낮음"].copy()

    if non_overfit_df.empty:
        candidate_df = results_df.copy()
    else:
        candidate_df = non_overfit_df.copy()

    stable_df = candidate_df[candidate_df["gap_roc_auc"] <= acceptable_roc_gap].copy()

    if not stable_df.empty:
        selected_df = stable_df.sort_values(
            by=["train_roc_auc", "test_roc_auc", "gap_roc_auc"],
            ascending=[False, False, True],
        )
    else:
        selected_df = candidate_df.sort_values(
            by=["gap_roc_auc", "test_roc_auc", "train_roc_auc"],
            ascending=[True, False, False],
        )

    return selected_df.index[0]


# 데이터 로드 및 병합부
df_list = [pd.read_csv(path) for path in file_paths]
df = pd.concat(df_list, ignore_index=True).copy()

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    if col in df.columns:
        df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 숫자형 변환부
for col in df.columns:
    if col in categorical_candidates or col in {"USER_KEY", "reg_date", "end_date"}:
        continue

    df[col] = pd.to_numeric(df[col], errors="coerce")

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수 컬럼 구성부
base_features = [
    col for col in base_feature_candidates
    if col in df.columns and col not in exclude_cols
]

extra_features = [
    col for col in df.columns
    if col not in exclude_cols and col not in base_features
]

feature_cols = base_features + extra_features

if not feature_cols:
    raise ValueError("사용 가능한 입력 변수 컬럼이 없습니다.")

# 입력 변수, 타깃 변수 생성부
X = df[feature_cols].copy()

# 양성 클래스 정의부
# is_repurchase == 0 예측 목표 설정부
y = (df["is_repurchase_num"] == 0).astype(int)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 숫자형, 범주형 컬럼 구분부
categorical_features = [
    col for col in feature_cols
    if col in categorical_candidates
]

numeric_features = [
    col for col in feature_cols
    if col not in categorical_features
]

# VIF 계산용 숫자형 데이터 준비부
if numeric_features:
    numeric_imputer = SimpleImputer(strategy="median")
    X_train_numeric = pd.DataFrame(
        numeric_imputer.fit_transform(X_train[numeric_features]),
        columns=numeric_features,
        index=X_train.index,
    )

    reduced_numeric_df, final_vif_df = remove_high_vif_features(
        X_train_numeric,
        threshold=vif_threshold,
    )

    final_numeric_features = reduced_numeric_df.columns.tolist()
else:
    final_vif_df = pd.DataFrame(columns=["feature", "vif"])
    final_numeric_features = []

# VIF 반영 후 최종 컬럼 구성부
final_feature_cols = final_numeric_features + categorical_features

if not final_feature_cols:
    raise ValueError("최종 사용 가능한 입력 변수 컬럼이 없습니다.")

# 누수 의심 컬럼 제거부
suspicious_features = get_suspicious_features(final_feature_cols)
safe_feature_cols = [
    col for col in final_feature_cols
    if col not in suspicious_features
]

if not safe_feature_cols:
    raise ValueError("누수 의심 컬럼 제거 후 사용 가능한 입력 변수 컬럼이 없습니다.")

# 모델 성능 평가부
results_df, fitted_models = evaluate_models(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=safe_feature_cols,
    categorical_candidates=categorical_candidates,
)

# 최적 모델 선택부
best_model_name = select_best_model(results_df)
best_pipeline = fitted_models[best_model_name]
best_model = best_pipeline.named_steps["model"]

# 누수 점검부
remaining_suspicious_features = get_suspicious_features(safe_feature_cols)
remaining_suspicious_count = len(remaining_suspicious_features)

shuffled_roc_auc = check_shuffled_target(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=safe_feature_cols,
    categorical_candidates=categorical_candidates,
    model=best_model,
)

leakage_judgment = judge_leakage(
    shuffled_roc_auc=shuffled_roc_auc,
    suspicious_feature_count=remaining_suspicious_count,
)

# 최적 모델 결과 추출부
best_model_row = results_df.loc[best_model_name]

# 출력부
print("양성 클래스 기준: is_repurchase == 0")
print()
print(f"VIF 반영 후 최종 사용 컬럼 수: {len(final_feature_cols)}")
print(f"누수 의심 컬럼 제거 수: {len(suspicious_features)}")
print(f"누수 의심 컬럼 제거 후 사용 컬럼 수: {len(safe_feature_cols)}")
print()
print("모델별 성능")
print(results_df.to_string())
print()
print("최종 선택 모델 결과")
print(f"선택 모델: {best_model_name}")
print(f"train_f1: {best_model_row['train_f1']:.4f}")
print(f"test_f1: {best_model_row['test_f1']:.4f}")
print(f"train_roc_auc: {best_model_row['train_roc_auc']:.4f}")
print(f"test_roc_auc: {best_model_row['test_roc_auc']:.4f}")
print(f"gap_roc_auc: {best_model_row['gap_roc_auc']:.4f}")
print(f"과적합 여부: {best_model_row['overfitting']}")
print()
print("데이터 누수 점검 결과")
print(f"남은 누수 의심 컬럼 수: {remaining_suspicious_count}")
print(f"셔플 타깃 ROC AUC: {shuffled_roc_auc:.4f}")
print(f"데이터 누수 가능성: {leakage_judgment}")


양성 클래스 기준: is_repurchase == 0

VIF 반영 후 최종 사용 컬럼 수: 73
누수 의심 컬럼 제거 수: 11
누수 의심 컬럼 제거 후 사용 컬럼 수: 62

모델별 성능
                    train_f1  test_f1  gap_f1  train_roc_auc  test_roc_auc  gap_roc_auc overfitting
model                                                                                              
LogisticRegression    0.6476   0.6462  0.0014         0.8439        0.8486      -0.0047          낮음
RandomForest          0.9772   0.6895  0.2877         0.9983        0.8874       0.1109          의심
GradientBoosting      0.6955   0.6697  0.0259         0.8997        0.8882       0.0115          낮음

최종 선택 모델 결과
선택 모델: GradientBoosting
train_f1: 0.6955
test_f1: 0.6697
train_roc_auc: 0.8997
test_roc_auc: 0.8882
gap_roc_auc: 0.0115
과적합 여부: 낮음

데이터 누수 점검 결과
남은 누수 의심 컬럼 수: 0
셔플 타깃 ROC AUC: 0.5114
데이터 누수 가능성: 낮음


## 여러 모델 비교

In [18]:
import sys
import importlib.util

print("현재 Python 실행 경로")
print(sys.executable)
print()

modules = ["xgboost", "lightgbm", "catboost"]

print("패키지 설치 여부 확인")
for module_name in modules:
    spec = importlib.util.find_spec(module_name)

    if spec is None:
        print(f"{module_name}: 설치 안 됨")
    else:
        print(f"{module_name}: 설치됨")

print()
print("패키지 import 및 버전 확인")

for module_name in modules:
    try:
        module = __import__(module_name)
        version = getattr(module, "__version__", "버전 확인 불가")
        print(f"{module_name}: import 성공 / version = {version}")
    except Exception as e:
        print(f"{module_name}: import 실패 / {e}")


현재 Python 실행 경로
c:\Users\user\anaconda3\python.exe

패키지 설치 여부 확인
xgboost: 설치됨
lightgbm: 설치됨
catboost: 설치됨

패키지 import 및 버전 확인
xgboost: import 성공 / version = 3.2.0
lightgbm: import 성공 / version = 4.6.0
catboost: import 성공 / version = 1.2.10


In [19]:
import re
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")


# 파일 경로 설정부
file_paths = [
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_0.csv",
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_1.csv",
]

# 기존 사용 컬럼 우선순위 설정부
base_feature_candidates = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
]

# 제외 컬럼 설정부
exclude_cols = {
    "USER_NUM",
    "USER_KEY",
    "reg_date",
    "end_date",
    "is_repurchase",
    "is_repurchase_num",
}

# 범주형 컬럼 후보 설정부
categorical_candidates = {
    "payment_device",
    "gender",
}

# 기준 설정부
vif_threshold = 10.0
acceptable_roc_gap = 0.03
shuffle_repeat = 3
random_state = 42


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# VIF 계산 함수부
def calculate_vif_table(df):
    rows = []

    for target_col in df.columns:
        x_cols = [col for col in df.columns if col != target_col]

        if not x_cols:
            vif_value = 1.0
        else:
            x = df[x_cols]
            y = df[target_col]

            model = LinearRegression()
            model.fit(x, y)
            r2 = model.score(x, y)

            if r2 >= 0.999999:
                vif_value = np.inf
            else:
                vif_value = 1.0 / (1.0 - r2)

        rows.append(
            {
                "feature": target_col,
                "vif": vif_value,
            }
        )

    return pd.DataFrame(rows).sort_values("vif", ascending=False).reset_index(drop=True)


# VIF 기반 제거 함수부
def remove_high_vif_features(df, threshold):
    working = df.copy()

    while working.shape[1] > 1:
        vif_df = calculate_vif_table(working)
        max_vif = vif_df["vif"].iloc[0]

        if pd.isna(max_vif) or max_vif < threshold:
            break

        drop_feature = vif_df.iloc[0]["feature"]
        working = working.drop(columns=[drop_feature])

    final_vif_df = calculate_vif_table(working)

    return working, final_vif_df


# 전처리기 생성 함수부
def build_preprocessor(feature_cols, categorical_candidates):
    categorical_features = [
        col for col in feature_cols
        if col in categorical_candidates
    ]

    numeric_features = [
        col for col in feature_cols
        if col not in categorical_features
    ]

    transformers = []

    if numeric_features:
        numeric_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]
        )
        transformers.append(("num", numeric_transformer, numeric_features))

    if categorical_features:
        categorical_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_onehot_encoder()),
            ]
        )
        transformers.append(("cat", categorical_transformer, categorical_features))

    return ColumnTransformer(transformers=transformers)


# 공통 지표 계산 함수부
def calculate_metrics(y_true, y_pred, y_proba):
    return {
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }


# 과적합 판정 함수부
def judge_overfitting(train_f1, test_f1, train_roc_auc, test_roc_auc):
    f1_gap = train_f1 - test_f1
    roc_auc_gap = train_roc_auc - test_roc_auc

    if f1_gap >= 0.05 or roc_auc_gap >= 0.05:
        return "의심"

    return "낮음"


# 셔플 타깃 위험도 판정 함수부
def judge_shuffled_target_risk(shuffled_roc_auc_mean):
    if shuffled_roc_auc_mean >= 0.60:
        return "위험"
    if shuffled_roc_auc_mean >= 0.56:
        return "주의"
    if shuffled_roc_auc_mean >= 0.53:
        return "약간 점검"
    return "낮음"


# 누수 의심 컬럼 추출 함수부
def get_suspicious_features(feature_cols):
    suspicious_patterns = {
        "repurchase": r"(^|_)repurchase($|_)",
        "churn": r"(^|_)churn($|_)",
        "last": r"(^|_)last($|_)",
        "end": r"(^|_)end($|_)",
        "gap": r"(^|_)gap($|_)",
        "retention": r"(^|_)retention($|_)",
        "recency": r"(^|_)recency($|_)",
        "late": r"(^|_)late($|_)",
        "week4": r"(^|_)week4($|_)",
        "week5": r"(^|_)week5($|_)",
    }

    suspicious_features = []

    for feature in feature_cols:
        lowered = feature.lower()

        matched = any(
            re.search(pattern, lowered)
            for pattern in suspicious_patterns.values()
        )

        if matched:
            suspicious_features.append(feature)

    return suspicious_features


# 셔플 타깃 평가 함수부
def evaluate_shuffled_target(
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    categorical_candidates,
    model,
    repeat_count,
):
    shuffled_scores = []

    for seed in range(repeat_count):
        shuffled_y_train = pd.Series(
            np.random.RandomState(random_state + seed).permutation(y_train.to_numpy()),
            index=y_train.index,
        )

        clf = Pipeline(
            steps=[
                (
                    "preprocessor",
                    build_preprocessor(feature_cols, categorical_candidates),
                ),
                ("model", clone(model)),
            ]
        )

        clf.fit(X_train[feature_cols], shuffled_y_train)
        y_test_proba = clf.predict_proba(X_test[feature_cols])[:, 1]
        shuffled_scores.append(roc_auc_score(y_test, y_test_proba))

    shuffled_mean = float(np.mean(shuffled_scores))
    shuffled_std = float(np.std(shuffled_scores))
    shuffled_min = float(np.min(shuffled_scores))
    shuffled_max = float(np.max(shuffled_scores))

    return {
        "shuffled_roc_auc_mean": shuffled_mean,
        "shuffled_roc_auc_std": shuffled_std,
        "shuffled_roc_auc_min": shuffled_min,
        "shuffled_roc_auc_max": shuffled_max,
        "shuffled_target_risk": judge_shuffled_target_risk(shuffled_mean),
    }


# 모델 사전 생성 함수부
def build_models():
    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=random_state,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(
            random_state=random_state,
        ),
    }

    missing_models = []

    try:
        from xgboost import XGBClassifier

        models["XGBoost"] = XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=random_state,
            n_jobs=-1,
            verbosity=0,
        )
    except Exception as e:
        missing_models.append(f"XGBoost 사용 불가: {e}")

    try:
        from lightgbm import LGBMClassifier

        models["LightGBM"] = LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=random_state,
            n_jobs=-1,
            verbosity=-1,
        )
    except Exception as e:
        missing_models.append(f"LightGBM 사용 불가: {e}")

    try:
        from catboost import CatBoostClassifier

        models["CatBoost"] = CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            loss_function="Logloss",
            eval_metric="AUC",
            random_state=random_state,
            verbose=0,
            allow_writing_files=False,
        )
    except Exception as e:
        missing_models.append(f"CatBoost 사용 불가: {e}")

    return models, missing_models


# 모델 평가 함수부
def evaluate_models(X_train, X_test, y_train, y_test, feature_cols, categorical_candidates):
    models, missing_models = build_models()

    results = []
    fitted_models = {}
    failed_models = []

    for model_name, model in models.items():
        try:
            clf = Pipeline(
                steps=[
                    (
                        "preprocessor",
                        build_preprocessor(feature_cols, categorical_candidates),
                    ),
                    ("model", clone(model)),
                ]
            )

            clf.fit(X_train[feature_cols], y_train)

            y_train_pred = clf.predict(X_train[feature_cols])
            y_train_proba = clf.predict_proba(X_train[feature_cols])[:, 1]

            y_test_pred = clf.predict(X_test[feature_cols])
            y_test_proba = clf.predict_proba(X_test[feature_cols])[:, 1]

            train_metrics = calculate_metrics(y_train, y_train_pred, y_train_proba)
            test_metrics = calculate_metrics(y_test, y_test_pred, y_test_proba)

            shuffled_result = evaluate_shuffled_target(
                X_train=X_train,
                X_test=X_test,
                y_train=y_train,
                y_test=y_test,
                feature_cols=feature_cols,
                categorical_candidates=categorical_candidates,
                model=model,
                repeat_count=shuffle_repeat,
            )

            results.append(
                {
                    "model": model_name,
                    "train_f1": train_metrics["f1_score"],
                    "test_f1": test_metrics["f1_score"],
                    "gap_f1": train_metrics["f1_score"] - test_metrics["f1_score"],
                    "train_roc_auc": train_metrics["roc_auc"],
                    "test_roc_auc": test_metrics["roc_auc"],
                    "gap_roc_auc": train_metrics["roc_auc"] - test_metrics["roc_auc"],
                    "overfitting": judge_overfitting(
                        train_f1=train_metrics["f1_score"],
                        test_f1=test_metrics["f1_score"],
                        train_roc_auc=train_metrics["roc_auc"],
                        test_roc_auc=test_metrics["roc_auc"],
                    ),
                    "shuffled_roc_auc_mean": shuffled_result["shuffled_roc_auc_mean"],
                    "shuffled_roc_auc_std": shuffled_result["shuffled_roc_auc_std"],
                    "shuffled_roc_auc_min": shuffled_result["shuffled_roc_auc_min"],
                    "shuffled_roc_auc_max": shuffled_result["shuffled_roc_auc_max"],
                    "shuffled_target_risk": shuffled_result["shuffled_target_risk"],
                }
            )

            fitted_models[model_name] = clf

        except Exception as e:
            failed_models.append(f"{model_name} 학습 실패: {e}")

    if not results:
        raise ValueError("학습에 성공한 모델이 없습니다.")

    results_df = (
        pd.DataFrame(results)
        .set_index("model")
        .sort_values("test_roc_auc", ascending=False)
        .round(4)
    )

    return results_df, fitted_models, missing_models, failed_models


# 최종 선택 모델 함수부
def select_best_model(results_df):
    candidate_df = results_df[results_df["overfitting"] == "낮음"].copy()

    if candidate_df.empty:
        candidate_df = results_df.copy()

    stable_df = candidate_df[candidate_df["gap_roc_auc"] <= acceptable_roc_gap].copy()

    if not stable_df.empty:
        selected_df = stable_df.sort_values(
            by=["train_roc_auc", "gap_roc_auc", "test_roc_auc"],
            ascending=[False, True, False],
        )
    else:
        selected_df = candidate_df.sort_values(
            by=["gap_roc_auc", "train_roc_auc", "test_roc_auc"],
            ascending=[True, False, False],
        )

    return selected_df.index[0]


# VIF 재점검 함수부
def check_remaining_high_vif(X_train, feature_cols, categorical_candidates, threshold):
    numeric_features = [
        col for col in feature_cols
        if col not in categorical_candidates
    ]

    if not numeric_features:
        return pd.DataFrame(columns=["feature", "vif"]), pd.DataFrame(columns=["feature", "vif"])

    numeric_imputer = SimpleImputer(strategy="median")
    X_train_numeric = pd.DataFrame(
        numeric_imputer.fit_transform(X_train[numeric_features]),
        columns=numeric_features,
        index=X_train.index,
    )

    vif_df = calculate_vif_table(X_train_numeric)
    high_vif_df = vif_df[vif_df["vif"] >= threshold].copy().reset_index(drop=True)

    return vif_df, high_vif_df


# 데이터 로드 및 병합부
df_list = [pd.read_csv(path) for path in file_paths]
df = pd.concat(df_list, ignore_index=True).copy()

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    if col in df.columns:
        df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 숫자형 변환부
for col in df.columns:
    if col in categorical_candidates or col in {"USER_KEY", "reg_date", "end_date"}:
        continue

    df[col] = pd.to_numeric(df[col], errors="coerce")

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수 컬럼 구성부
base_features = [
    col for col in base_feature_candidates
    if col in df.columns and col not in exclude_cols
]

extra_features = [
    col for col in df.columns
    if col not in exclude_cols and col not in base_features
]

feature_cols = base_features + extra_features

if not feature_cols:
    raise ValueError("사용 가능한 입력 변수 컬럼이 없습니다.")

# 입력 변수, 타깃 변수 생성부
X = df[feature_cols].copy()

# 양성 클래스 정의부
# is_repurchase == 0 예측 목표 설정부
y = (df["is_repurchase_num"] == 0).astype(int)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=random_state,
    stratify=y,
)

# 숫자형, 범주형 컬럼 구분부
categorical_features = [
    col for col in feature_cols
    if col in categorical_candidates
]

numeric_features = [
    col for col in feature_cols
    if col not in categorical_features
]

# VIF 계산용 숫자형 데이터 준비부
if numeric_features:
    numeric_imputer = SimpleImputer(strategy="median")
    X_train_numeric = pd.DataFrame(
        numeric_imputer.fit_transform(X_train[numeric_features]),
        columns=numeric_features,
        index=X_train.index,
    )

    reduced_numeric_df, final_vif_df = remove_high_vif_features(
        X_train_numeric,
        threshold=vif_threshold,
    )

    final_numeric_features = reduced_numeric_df.columns.tolist()
else:
    final_vif_df = pd.DataFrame(columns=["feature", "vif"])
    final_numeric_features = []

# VIF 반영 후 최종 컬럼 구성부
final_feature_cols = final_numeric_features + categorical_features

if not final_feature_cols:
    raise ValueError("최종 사용 가능한 입력 변수 컬럼이 없습니다.")

# 누수 의심 컬럼 제거부
suspicious_features = get_suspicious_features(final_feature_cols)
safe_feature_cols = [
    col for col in final_feature_cols
    if col not in suspicious_features
]

if not safe_feature_cols:
    raise ValueError("누수 의심 컬럼 제거 후 사용 가능한 입력 변수 컬럼이 없습니다.")

# 누수 의심 컬럼 제거 후 VIF 재점검부
safe_vif_df, safe_high_vif_df = check_remaining_high_vif(
    X_train=X_train,
    feature_cols=safe_feature_cols,
    categorical_candidates=categorical_candidates,
    threshold=vif_threshold,
)

# 모델 평가부
results_df, fitted_models, missing_models, failed_models = evaluate_models(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=safe_feature_cols,
    categorical_candidates=categorical_candidates,
)

# 최종 선택 모델 결정부
best_model_name = select_best_model(results_df)
best_model_row = results_df.loc[best_model_name]

# 출력부
print("양성 클래스 기준: is_repurchase == 0")
print()
print("컬럼 수 요약")
print(f"원본 후보 컬럼 수: {len(feature_cols)}")
print(f"VIF 반영 후 컬럼 수: {len(final_feature_cols)}")
print(f"누수 의심 컬럼 수: {len(suspicious_features)}")
print(f"최종 모델 사용 컬럼 수: {len(safe_feature_cols)}")
print(f"누수 의심 컬럼 제거 후 VIF 10 이상 컬럼 수: {len(safe_high_vif_df)}")
print()

print("모델별 성능 비교")
print(results_df.to_string())
print()

print("최종 선택 모델")
print(f"모델명: {best_model_name}")
print(f"train_f1: {best_model_row['train_f1']:.4f}")
print(f"test_f1: {best_model_row['test_f1']:.4f}")
print(f"gap_f1: {best_model_row['gap_f1']:.4f}")
print(f"train_roc_auc: {best_model_row['train_roc_auc']:.4f}")
print(f"test_roc_auc: {best_model_row['test_roc_auc']:.4f}")
print(f"gap_roc_auc: {best_model_row['gap_roc_auc']:.4f}")
print(f"과적합 여부: {best_model_row['overfitting']}")


양성 클래스 기준: is_repurchase == 0

컬럼 수 요약
원본 후보 컬럼 수: 134
VIF 반영 후 컬럼 수: 73
누수 의심 컬럼 수: 11
최종 모델 사용 컬럼 수: 62
누수 의심 컬럼 제거 후 VIF 10 이상 컬럼 수: 0

모델별 성능 비교
                    train_f1  test_f1  gap_f1  train_roc_auc  test_roc_auc  gap_roc_auc overfitting  shuffled_roc_auc_mean  shuffled_roc_auc_std  shuffled_roc_auc_min  shuffled_roc_auc_max shuffled_target_risk
model                                                                                                                                                                                                            
CatBoost              0.7468   0.7050  0.0418         0.9274        0.9003       0.0271          낮음                 0.5185                0.0098                0.5050                0.5278                   낮음
XGBoost               0.8641   0.7136  0.1505         0.9748        0.9002       0.0746          의심                 0.5213                0.0021                0.5184                0.5232                   낮음
LightGBM   

## payment_device와 gender 제거

In [21]:
import re
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")


# 파일 경로 설정부
file_paths = [
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_0.csv",
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_1.csv",
]

# 기존 사용 컬럼 우선순위 설정부
base_feature_candidates = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
]

# 제외 컬럼 설정부
exclude_cols = {
    "USER_NUM",
    "USER_KEY",
    "reg_date",
    "end_date",
    "is_repurchase",
    "is_repurchase_num",
}

# 범주형 컬럼 후보 설정부
categorical_candidates = {
    "payment_device",
    "gender",
}

# 기준 설정부
vif_threshold = 10.0
acceptable_roc_gap = 0.03
shuffle_repeat = 3
random_state = 42


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# VIF 계산 함수부
def calculate_vif_table(df):
    rows = []

    for target_col in df.columns:
        x_cols = [col for col in df.columns if col != target_col]

        if not x_cols:
            vif_value = 1.0
        else:
            x = df[x_cols]
            y = df[target_col]

            model = LinearRegression()
            model.fit(x, y)
            r2 = model.score(x, y)

            if r2 >= 0.999999:
                vif_value = np.inf
            else:
                vif_value = 1.0 / (1.0 - r2)

        rows.append(
            {
                "feature": target_col,
                "vif": vif_value,
            }
        )

    return pd.DataFrame(rows).sort_values("vif", ascending=False).reset_index(drop=True)


# VIF 기반 제거 함수부
def remove_high_vif_features(df, threshold):
    working = df.copy()

    while working.shape[1] > 1:
        vif_df = calculate_vif_table(working)
        max_vif = vif_df["vif"].iloc[0]

        if pd.isna(max_vif) or max_vif < threshold:
            break

        drop_feature = vif_df.iloc[0]["feature"]
        working = working.drop(columns=[drop_feature])

    final_vif_df = calculate_vif_table(working)

    return working, final_vif_df


# 전처리기 생성 함수부
def build_preprocessor(feature_cols, categorical_candidates):
    categorical_features = [
        col for col in feature_cols
        if col in categorical_candidates
    ]

    numeric_features = [
        col for col in feature_cols
        if col not in categorical_features
    ]

    transformers = []

    if numeric_features:
        numeric_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]
        )
        transformers.append(("num", numeric_transformer, numeric_features))

    if categorical_features:
        categorical_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_onehot_encoder()),
            ]
        )
        transformers.append(("cat", categorical_transformer, categorical_features))

    return ColumnTransformer(transformers=transformers)


# 공통 지표 계산 함수부
def calculate_metrics(y_true, y_pred, y_proba):
    return {
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }


# 과적합 판정 함수부
def judge_overfitting(train_f1, test_f1, train_roc_auc, test_roc_auc):
    f1_gap = train_f1 - test_f1
    roc_auc_gap = train_roc_auc - test_roc_auc

    if f1_gap >= 0.05 or roc_auc_gap >= 0.05:
        return "의심"

    return "낮음"


# 셔플 타깃 위험도 판정 함수부
def judge_shuffled_target_risk(shuffled_roc_auc_mean):
    if shuffled_roc_auc_mean >= 0.60:
        return "위험"
    if shuffled_roc_auc_mean >= 0.56:
        return "주의"
    if shuffled_roc_auc_mean >= 0.53:
        return "약간 점검"
    return "낮음"


# 누수 의심 컬럼 추출 함수부
def get_suspicious_features(feature_cols):
    suspicious_patterns = {
        "repurchase": r"(^|_)repurchase($|_)",
        "churn": r"(^|_)churn($|_)",
        "last": r"(^|_)last($|_)",
        "end": r"(^|_)end($|_)",
        "gap": r"(^|_)gap($|_)",
        "retention": r"(^|_)retention($|_)",
        "recency": r"(^|_)recency($|_)",
        "late": r"(^|_)late($|_)",
        "week4": r"(^|_)week4($|_)",
        "week5": r"(^|_)week5($|_)",
    }

    suspicious_features = []

    for feature in feature_cols:
        lowered = feature.lower()

        matched = any(
            re.search(pattern, lowered)
            for pattern in suspicious_patterns.values()
        )

        if matched:
            suspicious_features.append(feature)

    return suspicious_features


# 셔플 타깃 평가 함수부
def evaluate_shuffled_target(
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    categorical_candidates,
    model,
    repeat_count,
):
    shuffled_scores = []

    for seed in range(repeat_count):
        shuffled_y_train = pd.Series(
            np.random.RandomState(random_state + seed).permutation(y_train.to_numpy()),
            index=y_train.index,
        )

        clf = Pipeline(
            steps=[
                (
                    "preprocessor",
                    build_preprocessor(feature_cols, categorical_candidates),
                ),
                ("model", clone(model)),
            ]
        )

        clf.fit(X_train[feature_cols], shuffled_y_train)
        y_test_proba = clf.predict_proba(X_test[feature_cols])[:, 1]
        shuffled_scores.append(roc_auc_score(y_test, y_test_proba))

    shuffled_mean = float(np.mean(shuffled_scores))
    shuffled_std = float(np.std(shuffled_scores))
    shuffled_min = float(np.min(shuffled_scores))
    shuffled_max = float(np.max(shuffled_scores))

    return {
        "shuffled_roc_auc_mean": shuffled_mean,
        "shuffled_roc_auc_std": shuffled_std,
        "shuffled_roc_auc_min": shuffled_min,
        "shuffled_roc_auc_max": shuffled_max,
        "shuffled_target_risk": judge_shuffled_target_risk(shuffled_mean),
    }


# 모델 사전 생성 함수부
def build_models():
    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=random_state,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(
            random_state=random_state,
        ),
    }

    missing_models = []

    try:
        from xgboost import XGBClassifier

        models["XGBoost"] = XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=random_state,
            n_jobs=-1,
            verbosity=0,
        )
    except Exception as e:
        missing_models.append(f"XGBoost 사용 불가: {e}")

    try:
        from lightgbm import LGBMClassifier

        models["LightGBM"] = LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=random_state,
            n_jobs=-1,
            verbosity=-1,
        )
    except Exception as e:
        missing_models.append(f"LightGBM 사용 불가: {e}")

    try:
        from catboost import CatBoostClassifier

        models["CatBoost"] = CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            loss_function="Logloss",
            eval_metric="AUC",
            random_state=random_state,
            verbose=0,
            allow_writing_files=False,
        )
    except Exception as e:
        missing_models.append(f"CatBoost 사용 불가: {e}")

    return models, missing_models


# 모델 평가 함수부
def evaluate_models(X_train, X_test, y_train, y_test, feature_cols, categorical_candidates):
    models, missing_models = build_models()

    results = []
    fitted_models = {}
    failed_models = []

    for model_name, model in models.items():
        try:
            clf = Pipeline(
                steps=[
                    (
                        "preprocessor",
                        build_preprocessor(feature_cols, categorical_candidates),
                    ),
                    ("model", clone(model)),
                ]
            )

            clf.fit(X_train[feature_cols], y_train)

            y_train_pred = clf.predict(X_train[feature_cols])
            y_train_proba = clf.predict_proba(X_train[feature_cols])[:, 1]

            y_test_pred = clf.predict(X_test[feature_cols])
            y_test_proba = clf.predict_proba(X_test[feature_cols])[:, 1]

            train_metrics = calculate_metrics(y_train, y_train_pred, y_train_proba)
            test_metrics = calculate_metrics(y_test, y_test_pred, y_test_proba)

            shuffled_result = evaluate_shuffled_target(
                X_train=X_train,
                X_test=X_test,
                y_train=y_train,
                y_test=y_test,
                feature_cols=feature_cols,
                categorical_candidates=categorical_candidates,
                model=model,
                repeat_count=shuffle_repeat,
            )

            results.append(
                {
                    "model": model_name,
                    "train_f1": train_metrics["f1_score"],
                    "test_f1": test_metrics["f1_score"],
                    "gap_f1": train_metrics["f1_score"] - test_metrics["f1_score"],
                    "train_roc_auc": train_metrics["roc_auc"],
                    "test_roc_auc": test_metrics["roc_auc"],
                    "gap_roc_auc": train_metrics["roc_auc"] - test_metrics["roc_auc"],
                    "overfitting": judge_overfitting(
                        train_f1=train_metrics["f1_score"],
                        test_f1=test_metrics["f1_score"],
                        train_roc_auc=train_metrics["roc_auc"],
                        test_roc_auc=test_metrics["roc_auc"],
                    ),
                    "shuffled_roc_auc_mean": shuffled_result["shuffled_roc_auc_mean"],
                    "shuffled_roc_auc_std": shuffled_result["shuffled_roc_auc_std"],
                    "shuffled_roc_auc_min": shuffled_result["shuffled_roc_auc_min"],
                    "shuffled_roc_auc_max": shuffled_result["shuffled_roc_auc_max"],
                    "shuffled_target_risk": shuffled_result["shuffled_target_risk"],
                }
            )

            fitted_models[model_name] = clf

        except Exception as e:
            failed_models.append(f"{model_name} 학습 실패: {e}")

    if not results:
        raise ValueError("학습에 성공한 모델이 없습니다.")

    results_df = (
        pd.DataFrame(results)
        .set_index("model")
        .sort_values("test_roc_auc", ascending=False)
        .round(4)
    )

    return results_df, fitted_models, missing_models, failed_models


# 최종 선택 모델 함수부
def select_best_model(results_df):
    candidate_df = results_df[results_df["overfitting"] == "낮음"].copy()

    if candidate_df.empty:
        candidate_df = results_df.copy()

    stable_df = candidate_df[candidate_df["gap_roc_auc"] <= acceptable_roc_gap].copy()

    if not stable_df.empty:
        selected_df = stable_df.sort_values(
            by=["train_roc_auc", "gap_roc_auc", "test_roc_auc"],
            ascending=[False, True, False],
        )
    else:
        selected_df = candidate_df.sort_values(
            by=["gap_roc_auc", "train_roc_auc", "test_roc_auc"],
            ascending=[True, False, False],
        )

    return selected_df.index[0]


# VIF 재점검 함수부
def check_remaining_high_vif(X_train, feature_cols, categorical_candidates, threshold):
    numeric_features = [
        col for col in feature_cols
        if col not in categorical_candidates
    ]

    if not numeric_features:
        return pd.DataFrame(columns=["feature", "vif"]), pd.DataFrame(columns=["feature", "vif"])

    numeric_imputer = SimpleImputer(strategy="median")
    X_train_numeric = pd.DataFrame(
        numeric_imputer.fit_transform(X_train[numeric_features]),
        columns=numeric_features,
        index=X_train.index,
    )

    vif_df = calculate_vif_table(X_train_numeric)
    high_vif_df = vif_df[vif_df["vif"] >= threshold].copy().reset_index(drop=True)

    return vif_df, high_vif_df


# 시나리오 평가 함수부
def evaluate_feature_scenario(
    scenario_name,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    categorical_candidates,
    vif_threshold,
):
    if not feature_cols:
        raise ValueError(f"{scenario_name}에서 사용 가능한 입력 변수 컬럼이 없습니다.")

    scenario_vif_df, scenario_high_vif_df = check_remaining_high_vif(
        X_train=X_train,
        feature_cols=feature_cols,
        categorical_candidates=categorical_candidates,
        threshold=vif_threshold,
    )

    scenario_results_df, scenario_fitted_models, scenario_missing_models, scenario_failed_models = evaluate_models(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        feature_cols=feature_cols,
        categorical_candidates=categorical_candidates,
    )

    scenario_best_model_name = select_best_model(scenario_results_df)
    scenario_best_model_row = scenario_results_df.loc[scenario_best_model_name]

    return {
        "scenario_name": scenario_name,
        "feature_count": len(feature_cols),
        "vif_df": scenario_vif_df,
        "high_vif_df": scenario_high_vif_df,
        "results_df": scenario_results_df,
        "fitted_models": scenario_fitted_models,
        "missing_models": scenario_missing_models,
        "failed_models": scenario_failed_models,
        "best_model_name": scenario_best_model_name,
        "best_model_row": scenario_best_model_row,
    }


# 데이터 로드 및 병합부
df_list = [pd.read_csv(path) for path in file_paths]
df = pd.concat(df_list, ignore_index=True).copy()

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    if col in df.columns:
        df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 숫자형 변환부
for col in df.columns:
    if col in categorical_candidates or col in {"USER_KEY", "reg_date", "end_date"}:
        continue

    df[col] = pd.to_numeric(df[col], errors="coerce")

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수 컬럼 구성부
base_features = [
    col for col in base_feature_candidates
    if col in df.columns and col not in exclude_cols
]

extra_features = [
    col for col in df.columns
    if col not in exclude_cols and col not in base_features
]

feature_cols = base_features + extra_features

if not feature_cols:
    raise ValueError("사용 가능한 입력 변수 컬럼이 없습니다.")

# 입력 변수, 타깃 변수 생성부
X = df[feature_cols].copy()

# 양성 클래스 정의부
# is_repurchase == 0 예측 목표 설정부
y = (df["is_repurchase_num"] == 0).astype(int)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=random_state,
    stratify=y,
)

# 숫자형, 범주형 컬럼 구분부
categorical_features = [
    col for col in feature_cols
    if col in categorical_candidates
]

numeric_features = [
    col for col in feature_cols
    if col not in categorical_features
]

# VIF 계산용 숫자형 데이터 준비부
if numeric_features:
    numeric_imputer = SimpleImputer(strategy="median")
    X_train_numeric = pd.DataFrame(
        numeric_imputer.fit_transform(X_train[numeric_features]),
        columns=numeric_features,
        index=X_train.index,
    )

    reduced_numeric_df, final_vif_df = remove_high_vif_features(
        X_train_numeric,
        threshold=vif_threshold,
    )

    final_numeric_features = reduced_numeric_df.columns.tolist()
else:
    final_vif_df = pd.DataFrame(columns=["feature", "vif"])
    final_numeric_features = []

# VIF 반영 후 최종 컬럼 구성부
final_feature_cols = final_numeric_features + categorical_features

if not final_feature_cols:
    raise ValueError("최종 사용 가능한 입력 변수 컬럼이 없습니다.")

# 누수 의심 컬럼 제거부
suspicious_features = get_suspicious_features(final_feature_cols)
safe_feature_cols = [
    col for col in final_feature_cols
    if col not in suspicious_features
]

if not safe_feature_cols:
    raise ValueError("누수 의심 컬럼 제거 후 사용 가능한 입력 변수 컬럼이 없습니다.")

# gender, payment_device 추가 제거부
manual_remove_cols = {
    "gender",
    "payment_device",
}

safe_feature_cols_60 = [
    col for col in safe_feature_cols
    if col not in manual_remove_cols
]

if not safe_feature_cols_60:
    raise ValueError("gender, payment_device 제거 후 사용 가능한 입력 변수 컬럼이 없습니다.")

# 60개 컬럼 시나리오 평가부
scenario_60 = evaluate_feature_scenario(
    scenario_name="60개 컬럼",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=safe_feature_cols_60,
    categorical_candidates=categorical_candidates,
    vif_threshold=vif_threshold,
)

# 출력부
print("양성 클래스 기준: is_repurchase == 0")
print()
print("60개 컬럼 시나리오 요약")
print(f"VIF 반영 후 컬럼 수: {len(final_feature_cols)}")
print(f"누수 의심 컬럼 제거 후 컬럼 수: {len(safe_feature_cols)}")
print(f"gender, payment_device 제거 후 컬럼 수: {len(safe_feature_cols_60)}")
print(f"VIF 10 이상 컬럼 수: {len(scenario_60['high_vif_df'])}")
print()

print("60개 컬럼 시나리오 모델별 성능")
print(scenario_60["results_df"].to_string())
print()

print("60개 컬럼 시나리오 최종 선택 모델")
print(f"모델명: {scenario_60['best_model_name']}")
print(f"train_f1: {scenario_60['best_model_row']['train_f1']:.4f}")
print(f"test_f1: {scenario_60['best_model_row']['test_f1']:.4f}")
print(f"gap_f1: {scenario_60['best_model_row']['gap_f1']:.4f}")
print(f"train_roc_auc: {scenario_60['best_model_row']['train_roc_auc']:.4f}")
print(f"test_roc_auc: {scenario_60['best_model_row']['test_roc_auc']:.4f}")
print(f"gap_roc_auc: {scenario_60['best_model_row']['gap_roc_auc']:.4f}")
print(f"과적합 여부: {scenario_60['best_model_row']['overfitting']}")
print(f"셔플 타깃 ROC AUC 평균: {scenario_60['best_model_row']['shuffled_roc_auc_mean']:.4f}")
print(f"셔플 타깃 ROC AUC 표준편차: {scenario_60['best_model_row']['shuffled_roc_auc_std']:.4f}")
print(f"셔플 타깃 위험도: {scenario_60['best_model_row']['shuffled_target_risk']}")
print()

if scenario_60["missing_models"]:
    print("사용 불가 모델")
    for message in scenario_60["missing_models"]:
        print(message)
    print()

if scenario_60["failed_models"]:
    print("학습 실패 모델")
    for message in scenario_60["failed_models"]:
        print(message)
    print()

양성 클래스 기준: is_repurchase == 0

60개 컬럼 시나리오 요약
VIF 반영 후 컬럼 수: 73
누수 의심 컬럼 제거 후 컬럼 수: 62
gender, payment_device 제거 후 컬럼 수: 60
VIF 10 이상 컬럼 수: 0

60개 컬럼 시나리오 모델별 성능
                    train_f1  test_f1  gap_f1  train_roc_auc  test_roc_auc  gap_roc_auc overfitting  shuffled_roc_auc_mean  shuffled_roc_auc_std  shuffled_roc_auc_min  shuffled_roc_auc_max shuffled_target_risk
model                                                                                                                                                                                                            
XGBoost               0.8651   0.7234  0.1417         0.9755        0.9008       0.0746          의심                 0.5164                0.0034                0.5116                0.5188                   낮음
CatBoost              0.7440   0.7063  0.0376         0.9274        0.9004       0.0270          낮음                 0.5154                0.0144                0.4952                0.5279                   낮

In [27]:
import re
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


# 파일 경로 설정부
file_paths = [
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_0.csv",
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_1.csv",
]

# 기존 사용 컬럼 우선순위 설정부
base_feature_candidates = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
]

# 제외 컬럼 설정부
exclude_cols = {
    "USER_NUM",
    "USER_KEY",
    "reg_date",
    "end_date",
    "is_repurchase",
    "is_repurchase_num",
}

# 범주형 컬럼 후보 설정부
categorical_candidates = {
    "payment_device",
    "gender",
}

# 기준 설정부
corr_threshold = 0.90
vif_threshold = 10.0
acceptable_roc_gap = 0.03
random_state = 42


# 파일 로드 함수부
def load_feature_files(file_paths):
    df_list = []

    for file_path in file_paths:
        part_df = pd.read_csv(file_path).copy()

        if "is_promotion" not in part_df.columns:
            match = re.search(r"_(\d)\.csv$", file_path.lower())

            if match:
                part_df["is_promotion"] = int(match.group(1))
            else:
                raise ValueError(
                    f"is_promotion 컬럼이 없고 파일명에서도 그룹값을 추출할 수 없습니다: {file_path}"
                )

        df_list.append(part_df)

    return pd.concat(df_list, ignore_index=True).copy()


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# 상관관계 제거 함수부
def remove_high_correlation_features(df, threshold, protected_features=None):
    working_cols = list(df.columns)
    protected_features = set() if protected_features is None else set(protected_features)
    removed_rows = []

    while len(working_cols) > 1:
        corr_matrix = df[working_cols].corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

        pair_candidates = []

        for row_idx, row_name in enumerate(upper.index):
            for col_name in upper.columns[row_idx + 1:]:
                corr_value = upper.loc[row_name, col_name]

                if pd.notna(corr_value) and corr_value >= threshold:
                    pair_candidates.append(
                        {
                            "feature_a": row_name,
                            "feature_b": col_name,
                            "abs_corr": float(corr_value),
                        }
                    )

        if not pair_candidates:
            break

        pair_candidates = sorted(
            pair_candidates,
            key=lambda x: x["abs_corr"],
            reverse=True,
        )

        dropped_in_iteration = False

        for pair in pair_candidates:
            feature_a = pair["feature_a"]
            feature_b = pair["feature_b"]
            abs_corr = pair["abs_corr"]

            if feature_a not in working_cols or feature_b not in working_cols:
                continue

            if feature_a in protected_features and feature_b in protected_features:
                continue

            if feature_a in protected_features:
                keep_feature = feature_a
                drop_feature = feature_b
            elif feature_b in protected_features:
                keep_feature = feature_b
                drop_feature = feature_a
            else:
                index_a = working_cols.index(feature_a)
                index_b = working_cols.index(feature_b)

                if index_a <= index_b:
                    keep_feature = feature_a
                    drop_feature = feature_b
                else:
                    keep_feature = feature_b
                    drop_feature = feature_a

            removed_rows.append(
                {
                    "keep_feature": keep_feature,
                    "drop_feature": drop_feature,
                    "abs_corr": abs_corr,
                }
            )

            working_cols.remove(drop_feature)
            dropped_in_iteration = True
            break

        if not dropped_in_iteration:
            break

    removed_df = pd.DataFrame(removed_rows)
    reduced_df = df[working_cols].copy()

    return reduced_df, removed_df


# VIF 계산 함수부
def calculate_vif_table(df):
    rows = []

    for target_col in df.columns:
        x_cols = [col for col in df.columns if col != target_col]

        if not x_cols:
            vif_value = 1.0
        else:
            x = df[x_cols]
            y = df[target_col]

            model = LinearRegression()
            model.fit(x, y)
            r2 = model.score(x, y)

            if r2 >= 0.999999:
                vif_value = np.inf
            else:
                vif_value = 1.0 / (1.0 - r2)

        rows.append(
            {
                "feature": target_col,
                "vif": vif_value,
            }
        )

    return pd.DataFrame(rows).sort_values("vif", ascending=False).reset_index(drop=True)


# VIF 제거 함수부
def remove_high_vif_features(df, threshold, protected_features=None):
    working = df.copy()
    protected_features = set() if protected_features is None else set(protected_features)
    removed_rows = []

    while working.shape[1] > 1:
        vif_df = calculate_vif_table(working)
        max_vif = vif_df["vif"].iloc[0]

        if pd.isna(max_vif) or max_vif < threshold:
            break

        drop_feature = None
        drop_vif = None

        for _, row in vif_df.iterrows():
            candidate_feature = row["feature"]

            if candidate_feature not in protected_features:
                drop_feature = candidate_feature
                drop_vif = row["vif"]
                break

        if drop_feature is None:
            break

        removed_rows.append(
            {
                "drop_feature": drop_feature,
                "vif": drop_vif,
            }
        )

        working = working.drop(columns=[drop_feature])

    removed_df = pd.DataFrame(removed_rows)
    final_vif_df = calculate_vif_table(working)

    return working, removed_df, final_vif_df


# 전처리기 생성 함수부
def build_preprocessor(feature_cols, categorical_candidates):
    categorical_features = [
        col for col in feature_cols
        if col in categorical_candidates
    ]

    numeric_features = [
        col for col in feature_cols
        if col not in categorical_features
    ]

    transformers = []

    if numeric_features:
        numeric_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]
        )
        transformers.append(("num", numeric_transformer, numeric_features))

    if categorical_features:
        categorical_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_onehot_encoder()),
            ]
        )
        transformers.append(("cat", categorical_transformer, categorical_features))

    return ColumnTransformer(transformers=transformers)


# 공통 지표 계산 함수부
def calculate_metrics(y_true, y_pred, y_proba):
    return {
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }


# 과적합 판정 함수부
def judge_overfitting(train_f1, test_f1, train_roc_auc, test_roc_auc):
    f1_gap = train_f1 - test_f1
    roc_auc_gap = train_roc_auc - test_roc_auc

    if f1_gap >= 0.05 or roc_auc_gap >= 0.05:
        return "의심"

    return "낮음"


# 데이터 누수 위험도 판정 함수부
def judge_leakage_risk(remaining_suspicious_count):
    if remaining_suspicious_count > 0:
        return "점검 필요"
    return "낮음"


# 누수 의심 컬럼 추출 함수부
def get_suspicious_features(feature_cols):
    suspicious_patterns = {
        "repurchase": r"(^|_)repurchase($|_)",
        "churn": r"(^|_)churn($|_)",
        "last": r"(^|_)last($|_)",
        "end": r"(^|_)end($|_)",
        "gap": r"(^|_)gap($|_)",
        "retention": r"(^|_)retention($|_)",
        "recency": r"(^|_)recency($|_)",
        "late": r"(^|_)late($|_)",
        "week4": r"(^|_)week4($|_)",
        "week5": r"(^|_)week5($|_)",
    }

    suspicious_features = []

    for feature in feature_cols:
        lowered = feature.lower()

        matched = any(
            re.search(pattern, lowered)
            for pattern in suspicious_patterns.values()
        )

        if matched:
            suspicious_features.append(feature)

    return suspicious_features


# 모델 생성 함수부
def build_models():
    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=random_state,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(
            random_state=random_state,
        ),
    }

    try:
        from xgboost import XGBClassifier

        models["XGBoost"] = XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=random_state,
            n_jobs=-1,
            verbosity=0,
        )
    except Exception:
        pass

    try:
        from lightgbm import LGBMClassifier

        models["LightGBM"] = LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=random_state,
            n_jobs=-1,
            verbosity=-1,
        )
    except Exception:
        pass

    try:
        from catboost import CatBoostClassifier

        models["CatBoost"] = CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            loss_function="Logloss",
            eval_metric="AUC",
            random_state=random_state,
            verbose=0,
            allow_writing_files=False,
        )
    except Exception:
        pass

    return models


# 모델 평가 함수부
def evaluate_models(
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    categorical_candidates,
    leakage_risk,
):
    models = build_models()
    results = []

    for model_name, model in models.items():
        clf = Pipeline(
            steps=[
                (
                    "preprocessor",
                    build_preprocessor(feature_cols, categorical_candidates),
                ),
                ("model", clone(model)),
            ]
        )

        clf.fit(X_train[feature_cols], y_train)

        y_train_pred = clf.predict(X_train[feature_cols])
        y_train_proba = clf.predict_proba(X_train[feature_cols])[:, 1]

        y_test_pred = clf.predict(X_test[feature_cols])
        y_test_proba = clf.predict_proba(X_test[feature_cols])[:, 1]

        train_metrics = calculate_metrics(y_train, y_train_pred, y_train_proba)
        test_metrics = calculate_metrics(y_test, y_test_pred, y_test_proba)

        results.append(
            {
                "model": model_name,
                "train_f1": train_metrics["f1_score"],
                "test_f1": test_metrics["f1_score"],
                "gap_f1": train_metrics["f1_score"] - test_metrics["f1_score"],
                "train_roc_auc": train_metrics["roc_auc"],
                "test_roc_auc": test_metrics["roc_auc"],
                "gap_roc_auc": train_metrics["roc_auc"] - test_metrics["roc_auc"],
                "overfitting": judge_overfitting(
                    train_f1=train_metrics["f1_score"],
                    test_f1=test_metrics["f1_score"],
                    train_roc_auc=train_metrics["roc_auc"],
                    test_roc_auc=test_metrics["roc_auc"],
                ),
                "leakage_risk": leakage_risk,
            }
        )

    results_df = (
        pd.DataFrame(results)
        .set_index("model")
        .sort_values("test_roc_auc", ascending=False)
        .round(4)
    )

    return results_df


# 최종 선택 모델 함수부
def select_best_model(results_df):
    candidate_df = results_df[results_df["overfitting"] == "낮음"].copy()

    if candidate_df.empty:
        candidate_df = results_df.copy()

    stable_df = candidate_df[candidate_df["gap_roc_auc"] <= acceptable_roc_gap].copy()

    if not stable_df.empty:
        selected_df = stable_df.sort_values(
            by=["train_roc_auc", "gap_roc_auc", "test_roc_auc"],
            ascending=[False, True, False],
        )
    else:
        selected_df = candidate_df.sort_values(
            by=["gap_roc_auc", "train_roc_auc", "test_roc_auc"],
            ascending=[True, False, False],
        )

    return selected_df.index[0]


# 시나리오 컬럼 구성 함수부
def build_feature_scenario_set(
    X_train,
    original_feature_cols,
    categorical_candidates,
    corr_threshold,
    vif_threshold,
    protected_numeric_features=None,
    manual_remove_cols=None,
):
    manual_remove_cols = set() if manual_remove_cols is None else set(manual_remove_cols)
    protected_numeric_features = set() if protected_numeric_features is None else set(protected_numeric_features)

    categorical_features = [
        col for col in original_feature_cols
        if col in categorical_candidates
    ]

    numeric_features = [
        col for col in original_feature_cols
        if col not in categorical_features
    ]

    if numeric_features:
        numeric_imputer = SimpleImputer(strategy="median")
        X_train_numeric = pd.DataFrame(
            numeric_imputer.fit_transform(X_train[numeric_features]),
            columns=numeric_features,
            index=X_train.index,
        )

        corr_reduced_numeric_df, corr_removed_df = remove_high_correlation_features(
            X_train_numeric,
            threshold=corr_threshold,
            protected_features=protected_numeric_features,
        )

        vif_reduced_numeric_df, vif_removed_df, _ = remove_high_vif_features(
            corr_reduced_numeric_df,
            threshold=vif_threshold,
            protected_features=protected_numeric_features,
        )

        final_numeric_features = vif_reduced_numeric_df.columns.tolist()
        after_corr_numeric_count = corr_reduced_numeric_df.shape[1]
        after_vif_numeric_count = vif_reduced_numeric_df.shape[1]
    else:
        corr_removed_df = pd.DataFrame(columns=["keep_feature", "drop_feature", "abs_corr"])
        vif_removed_df = pd.DataFrame(columns=["drop_feature", "vif"])
        final_numeric_features = []
        after_corr_numeric_count = 0
        after_vif_numeric_count = 0

    feature_cols_after_numeric_filter = final_numeric_features + categorical_features

    suspicious_features = get_suspicious_features(feature_cols_after_numeric_filter)

    safe_feature_cols = [
        col for col in feature_cols_after_numeric_filter
        if col not in suspicious_features
    ]

    final_feature_cols = [
        col for col in safe_feature_cols
        if col not in manual_remove_cols
    ]

    remaining_suspicious_features = get_suspicious_features(final_feature_cols)

    return {
        "original_feature_count": len(original_feature_cols),
        "original_numeric_count": len(numeric_features),
        "corr_removed_count": len(corr_removed_df),
        "after_corr_numeric_count": after_corr_numeric_count,
        "vif_removed_count": len(vif_removed_df),
        "after_vif_numeric_count": after_vif_numeric_count,
        "suspicious_removed_count": len(suspicious_features),
        "manual_removed_count": len([col for col in manual_remove_cols if col in safe_feature_cols]),
        "final_feature_count": len(final_feature_cols),
        "final_feature_cols": final_feature_cols,
        "is_promotion_final": "is_promotion" in final_feature_cols,
        "remaining_suspicious_count": len(remaining_suspicious_features),
        "leakage_risk": judge_leakage_risk(len(remaining_suspicious_features)),
    }


# 데이터 로드 및 병합부
df = load_feature_files(file_paths)

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    if col in df.columns:
        df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 숫자형 변환부
for col in df.columns:
    if col in categorical_candidates or col in {"USER_KEY", "reg_date", "end_date"}:
        continue

    df[col] = pd.to_numeric(df[col], errors="coerce")

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수 컬럼 구성부
base_features = [
    col for col in base_feature_candidates
    if col in df.columns and col not in exclude_cols
]

extra_features = [
    col for col in df.columns
    if col not in exclude_cols and col not in base_features
]

feature_cols = base_features + extra_features

if not feature_cols:
    raise ValueError("사용 가능한 입력 변수 컬럼이 없습니다.")

# 입력 변수, 타깃 변수 생성부
X = df[feature_cols].copy()

# 양성 클래스 정의부
y = (df["is_repurchase_num"] == 0).astype(int)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=random_state,
    stratify=y,
)

# 수동 제거 컬럼 설정부
manual_remove_cols = {
    "gender",
    "payment_device",
}

# 기본 시나리오 컬럼 구성부
base_feature_set = build_feature_scenario_set(
    X_train=X_train,
    original_feature_cols=feature_cols,
    categorical_candidates=categorical_candidates,
    corr_threshold=corr_threshold,
    vif_threshold=vif_threshold,
    protected_numeric_features=None,
    manual_remove_cols=manual_remove_cols,
)

# is_promotion 포함 시나리오 컬럼 구성부
promo_feature_set = build_feature_scenario_set(
    X_train=X_train,
    original_feature_cols=feature_cols,
    categorical_candidates=categorical_candidates,
    corr_threshold=corr_threshold,
    vif_threshold=vif_threshold,
    protected_numeric_features={"is_promotion"},
    manual_remove_cols=manual_remove_cols,
)

# 모델 평가부
base_results_df = evaluate_models(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=base_feature_set["final_feature_cols"],
    categorical_candidates=categorical_candidates,
    leakage_risk=base_feature_set["leakage_risk"],
)

promo_results_df = evaluate_models(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=promo_feature_set["final_feature_cols"],
    categorical_candidates=categorical_candidates,
    leakage_risk=promo_feature_set["leakage_risk"],
)

# 최종 선택 모델 추출부
base_best_model = select_best_model(base_results_df)
promo_best_model = select_best_model(promo_results_df)

# 전처리 요약표 생성부
preprocess_summary_df = pd.DataFrame(
    [
        {
            "scenario": "기본 시나리오",
            "original_cols": base_feature_set["original_feature_count"],
            "numeric_cols": base_feature_set["original_numeric_count"],
            "corr_removed": base_feature_set["corr_removed_count"],
            "after_corr_num": base_feature_set["after_corr_numeric_count"],
            "vif_removed": base_feature_set["vif_removed_count"],
            "after_vif_num": base_feature_set["after_vif_numeric_count"],
            "suspicious_removed": base_feature_set["suspicious_removed_count"],
            "manual_removed": base_feature_set["manual_removed_count"],
            "final_cols": base_feature_set["final_feature_count"],
            "remaining_suspicious": base_feature_set["remaining_suspicious_count"],
            "leakage_risk": base_feature_set["leakage_risk"],
            "is_promotion_included": base_feature_set["is_promotion_final"],
        },
        {
            "scenario": "is_promotion 포함 시나리오",
            "original_cols": promo_feature_set["original_feature_count"],
            "numeric_cols": promo_feature_set["original_numeric_count"],
            "corr_removed": promo_feature_set["corr_removed_count"],
            "after_corr_num": promo_feature_set["after_corr_numeric_count"],
            "vif_removed": promo_feature_set["vif_removed_count"],
            "after_vif_num": promo_feature_set["after_vif_numeric_count"],
            "suspicious_removed": promo_feature_set["suspicious_removed_count"],
            "manual_removed": promo_feature_set["manual_removed_count"],
            "final_cols": promo_feature_set["final_feature_count"],
            "remaining_suspicious": promo_feature_set["remaining_suspicious_count"],
            "leakage_risk": promo_feature_set["leakage_risk"],
            "is_promotion_included": promo_feature_set["is_promotion_final"],
        },
    ]
)

# 성능 비교표 생성부
performance_df = pd.concat(
    [
        base_results_df.reset_index().assign(
            scenario="기본 시나리오",
            feature_count=base_feature_set["final_feature_count"],
        ),
        promo_results_df.reset_index().assign(
            scenario="is_promotion 포함 시나리오",
            feature_count=promo_feature_set["final_feature_count"],
        ),
    ],
    ignore_index=True,
)

performance_df = performance_df[
    [
        "scenario",
        "model",
        "feature_count",
        "train_f1",
        "test_f1",
        "gap_f1",
        "train_roc_auc",
        "test_roc_auc",
        "gap_roc_auc",
        "overfitting",
        "leakage_risk",
    ]
].sort_values(
    by=["scenario", "test_roc_auc"],
    ascending=[True, False],
).round(4)

# 최종 선택 모델 비교표 생성부
best_model_comparison_df = pd.DataFrame(
    [
        {
            "scenario": "기본 시나리오",
            "feature_count": base_feature_set["final_feature_count"],
            "best_model": base_best_model,
            "train_f1": base_results_df.loc[base_best_model, "train_f1"],
            "test_f1": base_results_df.loc[base_best_model, "test_f1"],
            "gap_f1": base_results_df.loc[base_best_model, "gap_f1"],
            "train_roc_auc": base_results_df.loc[base_best_model, "train_roc_auc"],
            "test_roc_auc": base_results_df.loc[base_best_model, "test_roc_auc"],
            "gap_roc_auc": base_results_df.loc[base_best_model, "gap_roc_auc"],
            "overfitting": base_results_df.loc[base_best_model, "overfitting"],
            "leakage_risk": base_results_df.loc[base_best_model, "leakage_risk"],
        },
        {
            "scenario": "is_promotion 포함 시나리오",
            "feature_count": promo_feature_set["final_feature_count"],
            "best_model": promo_best_model,
            "train_f1": promo_results_df.loc[promo_best_model, "train_f1"],
            "test_f1": promo_results_df.loc[promo_best_model, "test_f1"],
            "gap_f1": promo_results_df.loc[promo_best_model, "gap_f1"],
            "train_roc_auc": promo_results_df.loc[promo_best_model, "train_roc_auc"],
            "test_roc_auc": promo_results_df.loc[promo_best_model, "test_roc_auc"],
            "gap_roc_auc": promo_results_df.loc[promo_best_model, "gap_roc_auc"],
            "overfitting": promo_results_df.loc[promo_best_model, "overfitting"],
            "leakage_risk": promo_results_df.loc[promo_best_model, "leakage_risk"],
        },
    ]
).round(4)

# 출력부
print("양성 클래스 기준: is_repurchase == 0")
print("평가 방식: train_test_split 기반 단일 홀드아웃 평가")
print()

print("전처리 요약")
print(preprocess_summary_df.to_string(index=False))
print()

print("모델 성능 비교")
print(performance_df.to_string(index=False))
print()

print("최종 선택 모델 비교")
print(best_model_comparison_df.to_string(index=False))


양성 클래스 기준: is_repurchase == 0
평가 방식: train_test_split 기반 단일 홀드아웃 평가

전처리 요약
            scenario  original_cols  numeric_cols  corr_removed  after_corr_num  vif_removed  after_vif_num  suspicious_removed  manual_removed  final_cols  remaining_suspicious leakage_risk  is_promotion_included
             기본 시나리오            135           133            34              99           28             71                  10               2          61                     0           낮음                   True
is_promotion 포함 시나리오            135           133            34              99           28             71                  10               2          61                     0           낮음                   True

모델 성능 비교
            scenario              model  feature_count  train_f1  test_f1  gap_f1  train_roc_auc  test_roc_auc  gap_roc_auc overfitting leakage_risk
is_promotion 포함 시나리오           LightGBM             61    0.8502   0.7159  0.1343         0.9723        0.9020       0.0703